# Super-resolution demo (scale x3)
Загружает обученную модель, прогоняет одну картинку: слева — LR (пикселями), справа — результат модели.

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
from torchvision import transforms as T
from torchvision.transforms import functional as TF
import sys, os

sys.path.append(str(Path("../").resolve()))

from src.model import build_model
from src.utils import load_checkpoint
from src.metrics import PSNRMetric, SSIMMetric
from src.transforms.scale import ensure_divisible


In [ ]:
# Paths
ckpt_path = Path('../checkpoints/sr_unet_x3.pt')
image_path = Path('../data/sr/test_hr/0806.png')
scale = 3
crop_size = 192  # match validation crop
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(device)


In [ ]:
class Cfg: ...
config = Cfg()
config.model = Cfg(); config.model.in_channels=3; config.model.out_channels=3; config.model.base_channels=64; config.model.bilinear=True
model = build_model(config).to(device).eval()
state = load_checkpoint(ckpt_path, map_location=device)
model.load_state_dict(state['model_state_dict'])


In [ ]:
mean = [0.5, 0.5, 0.5]
std = [0.5, 0.5, 0.5]
normalize = T.Normalize(mean, std)
inv_normalize = T.Normalize(mean=[-m/s for m, s in zip(mean, std)], std=[1/s for s in std])

# Load HR and align with val pipeline (divisible by scale, center crop)
hr = Image.open(image_path).convert('RGB')
hr = ensure_divisible(hr, scale)
hr = TF.center_crop(hr, [crop_size, crop_size])

# Build LR -> upsample (same as dataset)
lr_small = hr.resize((hr.width // scale, hr.height // scale), Image.BICUBIC)
lr_pix = lr_small.resize(hr.size, Image.NEAREST)
lr_up = lr_small.resize(hr.size, Image.BICUBIC)

to_tensor = T.ToTensor()
lr_tensor = normalize(to_tensor(lr_up)).unsqueeze(0).to(device)
hr_tensor = normalize(to_tensor(hr)).unsqueeze(0).to(device)

with torch.no_grad():
    pred = model(lr_tensor).clamp(0, 1)

# Denormalize for visualization/metrics
pred_denorm = inv_normalize(pred[0].cpu()).clamp(0, 1)
hr_denorm = inv_normalize(hr_tensor[0].cpu()).clamp(0, 1)

psnr = PSNRMetric()(pred_denorm, hr_denorm)
ssim = SSIMMetric()(pred_denorm, hr_denorm)
print(f'PSNR: {psnr:.2f} dB, SSIM: {ssim:.4f}')

pred_img = T.ToPILImage()(pred_denorm)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(lr_pix); axes[0].set_title('LR (pixelated)'); axes[0].axis('off')
axes[1].imshow(pred_img); axes[1].set_title('Model SR'); axes[1].axis('off')
plt.tight_layout(); plt.show()
